# HTB network-log preprocessing: validate, drop, or impute

Companion to Module 292, Section 6. Run with the **AI** kernel. The CSV is HTB's 100-row demo dataset. This notebook keeps raw_data unchanged and makes separate drop and impute branches; it does not encode features, split data, or train a detector. Course behavior and safer alternatives are labeled distinctly.

## Load the same course CSV

HTB assumes demo_dataset.csv was extracted already. This self-contained notebook accepts a local demo_dataset.zip (or HTB_DEMO_DATASET_ZIP path) and otherwise fetches only the linked HTB ZIP in memory. It checks the expected member and size. No archive is extracted into the repository.

In [ ]:
from io import BytesIO
import os
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import numpy as np
import requests

url = 'https://cdn.services-k8s.prod.aws.htb.systems/content/modules/292/demo_dataset.zip'
zip_path = Path(os.environ.get('HTB_DEMO_DATASET_ZIP', 'demo_dataset.zip'))
if zip_path.is_file():
    archive_bytes = zip_path.read_bytes()
else:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    archive_bytes = response.content
with ZipFile(BytesIO(archive_bytes)) as archive:
    member = archive.getinfo('demo_dataset.csv')
    if member.file_size > 10_000_000:
        raise ValueError('Unexpectedly large course CSV')
    with archive.open(member) as source:
        raw_data = pd.read_csv(source)
required = {'log_id', 'source_ip', 'destination_port', 'protocol',
            'bytes_transferred', 'threat_level'}
assert required.issubset(raw_data.columns)
print('Raw rows:', len(raw_data))


## Check the five fields HTB names

HTB's checks are: an IPv4-shaped source address; an integer port from 0 through 65,535; a protocol in its allowlist; a nonnegative integer byte count; and a threat label in {0,1,2}. Read 0 ≤ port ≤ 65535 as **zero is less than or equal to port, and port is less than or equal to sixty-five thousand five hundred thirty-five**. The code explicitly rejects nulls and conversion errors. The IPv4 regular expression follows HTB's teaching approach; a production parser and explicit IPv6 policy would be stronger.

In [ ]:
import re

ipv4_pattern = re.compile(
    r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}'
    r'(25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)$'
)
valid_protocols = {'TCP', 'TLS', 'SSH', 'POP3', 'DNS',
                   'HTTPS', 'SMTP', 'FTP', 'UDP', 'HTTP'}

def valid_ip(value):
    return pd.notna(value) and bool(ipv4_pattern.fullmatch(str(value)))

def integer_in_range(value, lower, upper):
    try:
        number = int(value)  # Malformed text such as STRING_PORT raises ValueError.
    except (TypeError, ValueError, OverflowError):
        return False
    return lower <= number <= upper

assert valid_ip('192.168.1.10') and not valid_ip('256.1.1.1')
assert integer_in_range(443, 0, 65535)
assert not integer_in_range(70000, 0, 65535)

invalid = pd.DataFrame({
    'source_ip': ~raw_data['source_ip'].map(valid_ip),
    'destination_port': ~raw_data['destination_port'].map(
        lambda value: integer_in_range(value, 0, 65535)),
    'protocol': ~raw_data['protocol'].isin(valid_protocols),
    'bytes_transferred': ~raw_data['bytes_transferred'].map(
        lambda value: integer_in_range(value, 0, float('inf'))),
    'threat_level': ~raw_data['threat_level'].map(
        lambda value: integer_in_range(value, 0, 2)),
}, index=raw_data.index)
print('Failed checks per field (a row can fail more than one):')
print(invalid.sum())
print('Unique invalid rows:', int(invalid.any(axis=1).sum()))
assert invalid.shape == (len(raw_data), 5)


## Branch A — drop every invalid row

HTB drops indexes found by each validator. The same row may occur on several invalid lists; errors='ignore' lets later drops skip an index already removed. Using the union mask below is equivalent and makes the row count explicit. This sacrifices coverage to keep only fully valid rows; it is **not** always the right choice when malformed events may themselves be attacks.

In [ ]:
drop_clean = raw_data.loc[~invalid.any(axis=1)].copy()
print('Rows after dropping any invalid entry:', len(drop_clean))
print(drop_clean.describe(include='all'))
assert len(drop_clean) + int(invalid.any(axis=1).sum()) == len(raw_data)
if len(raw_data) == 100:
    assert len(drop_clean) == 77  # HTB's stated count for this course sample.


## Branch B — mark unknowns, then impute

HTB converts tokens such as STRING_PORT and ? to NaN (read **not a number**) and uses median values for numeric columns and the mode for protocol. The median of [100, 200, 9000] is 200; the mode of [TCP, TCP, UDP] is TCP. We also mark out-of-range numeric values and unrecognized protocols missing before filling them. **Course demonstration:** its numeric list includes threat_level, the answer label. That invents labels, so the resulting course_imputed table is shown for understanding only and must not be used to train or score a model.

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

unknown_tokens = ['INVALID_IP', 'MISSING_IP', 'STRING_PORT',
                  'UNUSED_PORT', 'NON_NUMERIC', 'NEGATIVE', '?']
marked = raw_data.replace(unknown_tokens, np.nan).copy()  # Never mutate raw_data.
for column in ('destination_port', 'bytes_transferred', 'threat_level'):
    marked[column] = pd.to_numeric(marked[column], errors='coerce')
marked['source_ip'] = marked['source_ip'].where(marked['source_ip'].map(valid_ip))
marked['protocol'] = marked['protocol'].where(marked['protocol'].isin(valid_protocols))
marked['destination_port'] = marked['destination_port'].where(
    marked['destination_port'].between(0, 65535))
marked['bytes_transferred'] = marked['bytes_transferred'].where(
    marked['bytes_transferred'] >= 0)
marked['threat_level'] = marked['threat_level'].where(
    marked['threat_level'].isin([0, 1, 2]))
print('Missing after marking invalid entries:')
print(marked.isna().sum())


In [ ]:
course_imputed = marked.copy()  # HTB-style demonstration; NOT for training.
course_numeric = ['destination_port', 'bytes_transferred', 'threat_level']
course_imputed[course_numeric] = SimpleImputer(strategy='median').fit_transform(
    course_imputed[course_numeric])
course_imputed[['protocol']] = SimpleImputer(strategy='most_frequent').fit_transform(
    course_imputed[['protocol']])
course_imputed['source_ip'] = course_imputed['source_ip'].fillna('0.0.0.0')
course_imputed['destination_port'] = course_imputed['destination_port'].clip(0, 65535)
print('HTB-style imputed table (illustration only):')
print(course_imputed.describe(include='all'))
print('Original unknown labels:', int(marked['threat_level'].isna().sum()))
print('Unknown labels after course-style imputation:',
      int(course_imputed['threat_level'].isna().sum()))


## Feature-only alternative and KNN comparison

For supervised learning, unknown target labels should remain unknown rather than become a guessed 0/1/2. The next branch imputes feature columns only and filters to truly labeled rows *afterward*. It is still a whole-dataset teaching example: a real training pipeline would split first, fit imputers on training rows only, and transform held-out rows using the training statistics. KNN means **k nearest neighbors**; here k=5. Run it on the still-missing feature columns, not after SimpleImputer has already filled every gap. Because port and byte counts have different scales, raw-distance neighbors may be misleading.

In [ ]:
feature_only = marked.copy()
numeric_features = ['destination_port', 'bytes_transferred']
feature_only[numeric_features] = SimpleImputer(strategy='median').fit_transform(
    feature_only[numeric_features])
feature_only[['protocol']] = SimpleImputer(strategy='most_frequent').fit_transform(
    feature_only[['protocol']])
labeled_rows = feature_only['threat_level'].isin([0, 1, 2])
supervised_candidates = feature_only.loc[labeled_rows].copy()
print('Rows with original known labels:', len(supervised_candidates))
print('Unknown labels deliberately retained:', int(feature_only['threat_level'].isna().sum()))

# Separate comparison, not a second imputation of already-filled course_imputed.
knn_features = KNNImputer(n_neighbors=5).fit_transform(marked[numeric_features])
print('KNN-imputed feature matrix shape:', knn_features.shape)
assert feature_only['threat_level'].isna().sum() == marked['threat_level'].isna().sum()


Neither branch is a finished intrusion detector. Dropping can remove security-relevant malformed traffic; imputation creates estimates rather than recovering ground truth. 0.0.0.0 is a missing-source sentinel, not the actual origin host, and clipping an invalid port can create a plausible but false value. Keep source records and invalidity indicators for audit, and decide downstream features/splits only in the later course sections.